# Proyecto final de aprendizaje automático

In [19]:
import pandas as pd
import sqlite3

## Paso 1: Definición del problema

### Objetivo: 
- Predecir la demanda futura de productos farmacéuticos para optimizar la gestión de inventario.
- Evitar quiebres de stock y sobrestock. 
- Reducir pérdida por vencimientos y mejorar disponibilidad.
##### Plan B: Predecir el total que abonará el cliente en una transacción, considerando cantidad, producto, rubro, cobertura, descuentos e impuestos.

## Paso 2: Obtencion y carga del conjunto de datos

In [20]:
df = pd.read_csv('../data/raw/farmacia-datos.csv', sep=';',encoding='latin-1')
df.head(12)

,Fecha,Tipo Mov.,Fac. Tipo,Fac. Suc.,Fac. Nun.,Fisc. Numero,Tipo Pago,Cant.,Precio,Producto,Sub. Total,Rubro,Cobertura,Ajustes,Desc. Adic.,Total. Cliente,IVA,Tasa Iva,Total Gravado,Total sin Gravar
0,01/01/25 01.13.46,F,B,0,379923,NaN,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,01/01/25 01.59.21,F,B,0,379924,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,01/01/25 02.01.59,F,B,0,379925,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,01/01/25 02.04.14,F,B,0,379926,NaN,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,01/01/25 02.08.35,F,B,0,379927,NaN,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"
5,01/01/25 06.06.42,F,B,0,379928,NaN,E,1,"24480,51",EVATEST DIGITAL env.x 1,"24480,51",FARMACIA,0,0,0,"24480,51","4248,683554",21,"20231,82645",0
6,01/01/25 06.22.37,F,B,0,379929,NaN,E,1,4800,GEN LP KETOROLAC blister 20 mg x 10,4800,FARMACIA,0,0,0,4800,0,0,0,4800
7,01/01/25 06.22.37,F,B,0,379929,NaN,E,1,4961,CAFIASPIRINA PLUS comp.x 20,4961,FARMACIA,0,0,0,4961,0,0,0,4961
8,01/01/25 07.46.36,F,B,0,379930,NaN,E,2,3500,ACCESORIO (unica),7000,FARMACIA,0,0,0,7000,"1214,876033",21,"5785,123967",0
9,02/01/25 08.08.38,F,B,0,379931,NaN,E,1,3430,QURA PLUS comp.rec.x 20,3430,FARMACIA,0,0,343,3087,0,0,0,3430


> Los datos provienen de un archivo Excel extraído del sistema de facturación de una farmacia real, convertido posteriormente a formato CSV para su procesamiento.

# Paso 3: Almacenar la información

In [21]:
# Create connection to SQLite
conn = sqlite3.connect('farmacia-datos.db')

In [22]:
# Save table
df.to_sql('ventas', conn, if_exists='replace', index=False)

117415

In [23]:
# Verify that the table exists
query = 'SELECT * FROM ventas LIMIT 5;'
pd.read_sql(query, conn)

,Fecha,Tipo Mov.,Fac. Tipo,Fac. Suc.,Fac. Nun.,Fisc. Numero,Tipo Pago,Cant.,Precio,Producto,Sub. Total,Rubro,Cobertura,Ajustes,Desc. Adic.,Total. Cliente,IVA,Tasa Iva,Total Gravado,Total sin Gravar
0,01/01/25 01.13.46,F,B,0,379923,None,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,9344,FARMACIA,0,0,0,9344,0,0,0,9344
1,01/01/25 01.59.21,F,B,0,379924,None,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"3032,85",0,0,"5142,11",0,0,0,"8174,96"
2,01/01/25 02.01.59,F,B,0,379925,None,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,"8174,96",FARMACIA,"5318,1",0,0,"2856,86",0,0,0,"8174,96"
3,01/01/25 02.04.14,F,B,0,379926,None,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,"8086,82",FARMACIA,0,0,0,"8086,82",0,0,0,"8086,82"
4,01/01/25 02.08.35,F,B,0,379927,None,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,"26750,12",FARMACIA,0,0,0,"26750,12",0,0,0,"26750,12"


Top 10 productos más vendidos

In [33]:
# Execute a real SQL query
query = '''
SELECT Producto, SUM('Cant.') AS Demanda_total
FROM ventas
GROUP BY Producto
ORDER BY Demanda_total DESC
LIMIT 10;
'''

pd.read_sql(query, conn)

,Producto,Demanda_total
0,ÓLEO Calcáreo x 500 ml.,0.0
1,ÓLEO Calcáreo x 240 ml.,0.0
2,º,0.0
3,|,0.0
4,q,0.0
5,esmalte,0.0
6,ZYPRED susp.oft.x 6 ml,0.0
7,ZUNDIC 5 mg comp.x 30,0.0
8,ZOXX 50 50 mg comp.rec.x 60,0.0
9,ZOPIROL 0.50% sol.oft.x 5 ml,0.0


> Se almacenaron los datos en una base de datos SQLite y se realizaron consultas SQL desde Python para identificar productos de alta rotación y patrones de demanda.

**Demanda total por producto**

In [31]:
query_product = '''
SELECT Producto, SUM("Cant.") AS Demanda_total
FROM ventas
GROUP BY Producto;
'''

pd.read_sql(query_product, conn)

,Producto,Demanda_total
0,None,1
1,CHUP.MANZANITA(C/DIBUJO)T/SILIC.+3M RED - ES...,1
2,(unica),4
3,1,1
4,1 1,7
...,...,...
7671,q,1
7672,|,1
7673,º,3
7674,ÓLEO Calcáreo x 240 ml.,1


Esta consulta permite identificar los productos con mayor volumen de ventas acumuladas, fundamentales para la gestión de stock y la priorización de reposición.

In [ ]:
query_top_20 = '''

'''

pd.read_sql(query_top_20, conn)

,Producto,Demanda_total
0,ÓLEO Calcáreo x 500 ml.,0.0
1,ÓLEO Calcáreo x 240 ml.,0.0
2,º,0.0
3,|,0.0
4,q,0.0
5,esmalte,0.0
6,ZYPRED susp.oft.x 6 ml,0.0
7,ZUNDIC 5 mg comp.x 30,0.0
8,ZOXX 50 50 mg comp.rec.x 60,0.0
9,ZOPIROL 0.50% sol.oft.x 5 ml,0.0


IndentationError: unexpected indent (492363088.py, line 2)

In [ ]:
df.shape

(117415, 20)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117415 entries, 0 to 117414
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Fecha             117415 non-null  object 
 1   Tipo Mov.         117415 non-null  object 
 2   Fac. Tipo         117415 non-null  object 
 3   Fac. Suc.         117415 non-null  int64  
 4   Fac. Nun.         117415 non-null  int64  
 5   Fisc. Numero      0 non-null       float64
 6   Tipo Pago         117415 non-null  object 
 7   Cant.             117415 non-null  int64  
 8   Precio            117415 non-null  object 
 9   Producto          117414 non-null  object 
 10  Sub. Total        117415 non-null  object 
 11  Rubro             117415 non-null  object 
 12  Cobertura         117415 non-null  object 
 13  Ajustes           117415 non-null  int64  
 14  Desc. Adic.       117415 non-null  object 
 15  Total. Cliente    117415 non-null  object 
 16  IVA               11

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Fac. Suc.,117415.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Fac. Nun.,117415.0,417643.591407,22420.160070,379923.0,398031.0,416281.0,437618.5,455865.0
Fisc. Numero,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cant.,117415.0,1.368130,1.862115,0.0,1.0,1.0,1.0,106.0
Ajustes,117415.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
Tasa Iva,117415.0,4.090363,8.316679,0.0,0.0,0.0,0.0,21.0


In [ ]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%d/%m/%y %H.%M.%S')

df['Year'] = df['Fecha'].dt.year
df['Month'] = df['Fecha'].dt.month
df['Week_day'] = df['Fecha'].dt.day_name()
df['Hour'] = df['Fecha'].dt.hour

In [ ]:
df.head(5)

,Fecha,Tipo Mov.,Fac. Tipo,Fac. Suc.,Fac. Nun.,Fisc. Numero,Tipo Pago,Cant.,Precio,Producto,...,Desc. Adic.,Total. Cliente,IVA,Tasa Iva,Total Gravado,Total sin Gravar,Year,Month,Week_day,Hour
0,2025-01-01 01:13:46,F,B,0,379923,NaN,E,1,9344,ACTRON PEDIATRICO 4% susp.oral x 100 ml,...,0,9344,0,0,0,9344,2025,1,Wednesday,1
1,2025-01-01 01:59:21,F,B,0,379924,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,...,0,"5142,11",0,0,0,"8174,96",2025,1,Wednesday,1
2,2025-01-01 02:01:59,F,B,0,379925,NaN,E,1,"8174,96",AMOXIDAL 500 mg comp.rec.x 21,...,0,"2856,86",0,0,0,"8174,96",2025,1,Wednesday,2
3,2025-01-01 02:04:14,F,B,0,379926,NaN,E,1,"8086,82",MUELITA FORTE GEL gel pomo x 10 g,...,0,"8086,82",0,0,0,"8086,82",2025,1,Wednesday,2
4,2025-01-01 02:08:35,F,B,0,379927,NaN,E,1,"26750,12",DIOXAFLEX B12 comp.x 20,...,0,"26750,12",0,0,0,"26750,12",2025,1,Wednesday,2


In [ ]:
numerical_variables = ['']

In [ ]:
categorical_variables = ['']